<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m2_03_bias_variance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M2.03 — One hundred laboratories

In the lab we met two ways for a model to be wrong: **bias** (it is too
rigid and misses the pattern, whatever data it gets) and **variance** (it
is so flexible that its predictions depend on which samples it happened
to be trained on). The slides showed drawings. In this workbook you
**measure** both on the concrete data, watch what averaging does to them,
and find the part of the error that no model can remove.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "averaging_reduces": "7c0766a4cdd8aa64",
    "higher_bias": "9f6183312a353732",
    "higher_variance": "97b910e393d8095b",
    "more_data_reduces": "7c0766a4cdd8aa64",
    "shallow_tree_problem": "ff7772053abf7d81",
    "smallest_possible_error": "ef2d127de37b942b"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. One hundred laboratories

In real life you train a model once, on the one dataset you have. To see
variance we need to train **the same kind of model many times on
different data** and compare the predictions.

So we play a game. We put 200 concrete samples aside: these are the
**test samples**, the same for everyone. The other 830 form a pool.
Then we imagine **100 laboratories**. Each one draws its own 300 samples
from the pool and trains its own model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

concrete = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/concrete_strength.csv"
)
data = concrete.drop(columns="strength")
target = concrete["strength"]
pool_data, test_data, pool_target, test_target = train_test_split(
    data, target, test_size=200, random_state=0
)


def draw_laboratories(n_laboratories=100, n_samples=300, seed=0):
    """For each laboratory, the positions of its samples in the pool."""
    rng = np.random.default_rng(seed)
    return [
        rng.choice(len(pool_data), size=n_samples, replace=False)
        for _ in range(n_laboratories)
    ]


def train_everywhere(model, laboratories):
    """Train a copy of `model` in each laboratory; predict the test samples.

    Returns a table with one row per laboratory and one column per test
    sample.
    """
    predictions = []
    for rows in laboratories:
        model.fit(pool_data.iloc[rows], pool_target.iloc[rows])
        predictions.append(model.predict(test_data))
    return np.array(predictions)

✏️ Give each laboratory **300** samples.

In [ ]:
n_samples = ____
laboratories = draw_laboratories(n_samples=n_samples)
print(f"{len(laboratories)} laboratories, {len(laboratories[0])} samples each, {len(test_data)} test samples")

Every laboratory trains two models, the same two as at the end of
workbook M2.02:

- a **linear regression**: rigid, one weight per feature;
- a **deep decision tree**, with no limit on its size: very flexible.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

predictions = {
    "linear regression": train_everywhere(LinearRegression(), laboratories),
    "deep tree": train_everywhere(DecisionTreeRegressor(random_state=0), laboratories),
}
predictions["deep tree"].shape

## 2. Seeing bias and variance

We pick six test samples, from weak to strong concrete. For each one we
have 100 predictions per kind of model: one from every laboratory.

🔮 **Predict first.** For which model will the 100 laboratories
**disagree the most** about the strength of a given sample? `"linear
regression"` or `"deep tree"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["linear regression", "deep tree"])

In [ ]:
order = np.argsort(test_target.to_numpy())
shown = order[np.linspace(10, len(order) - 11, 6).astype(int)]
truth = test_target.to_numpy()


def plot_disagreement(names):
    fig, axes = plt.subplots(1, len(names), figsize=(5.5 * len(names), 3.8), sharey=True)
    for ax, name in zip(axes, names):
        ax.boxplot(predictions[name][:, shown], whis=(0, 100))
        ax.plot(range(1, 7), truth[shown], "r*", markersize=12, label="true strength")
        ax.set(title=f"100 laboratories, {name}", xlabel="test sample")
    axes[0].set_ylabel("predicted strength (MPa)")
    axes[0].legend(loc="upper left")


plot_disagreement(["linear regression", "deep tree"])

Each box shows where the 100 predictions for one sample fall (the
whiskers go from the lowest to the highest). The red star is the true
strength.

- A **tall box** means the laboratories disagree: the prediction depends
  on which 300 samples a laboratory happened to draw. That is
  **variance**.
- A box whose **middle is far from the star** means that the laboratories
  are wrong together, in the same direction. That is **bias**.

Let us put numbers on it, over all 200 test samples:

- **bias**: how far the *average* prediction of the 100 laboratories is
  from the truth;
- **spread**: how much the predictions of the laboratories differ from
  one another (their standard deviation).

In [ ]:
def summarise(name):
    table = predictions[name]
    return {
        "bias (MPa)": np.abs(table.mean(axis=0) - truth).mean(),
        "spread (MPa)": table.std(axis=0).mean(),
        "error of one laboratory (MPa)": np.abs(table - truth).mean(),
    }


summary = pd.DataFrame({name: summarise(name) for name in predictions}).T
summary.round(2)

✏️ Which model has the higher **bias**, and which has the higher
**variance** (spread)? Write `"linear regression"` or `"deep tree"`.

In [ ]:
higher_bias = ____
higher_variance = ____
check("higher_bias", higher_bias)
check("higher_variance", higher_variance)

This is what the two words of the previous workbooks mean:

| | the model is… | its main problem |
|---|---|---|
| **underfitting** | too rigid | bias |
| **overfitting** | too dependent on its training set | variance |

🧠 **Think.** Look at the last column. The deep tree has the larger
spread, and yet one laboratory using it makes a smaller error than one
laboratory using the linear regression. How is that possible?

### ➕ Going further: is a small tree a stable model?

A tree limited to `max_depth=2` asks two questions in a row and can give
only 4 different answers. It is about as simple as a model gets.

🔮 **Predict first.** Will the **spread** of this shallow tree be closer
to the spread of the `"linear regression"` or to that of the `"deep
tree"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["linear regression", "deep tree"])

In [ ]:
predictions["shallow tree"] = train_everywhere(DecisionTreeRegressor(max_depth=2, random_state=0), laboratories)
summary = pd.DataFrame({name: summarise(name) for name in predictions}).T
summary.round(2)

✏️ Which problems does the shallow tree have? Write `"bias"`,
`"variance"` or `"both"`.

In [ ]:
shallow_tree_problem = ____
check("shallow_tree_problem", shallow_tree_problem)

"Simple" does not mean "stable". A tree chooses the threshold of its
first question from the data ("age below 21 days?" in one laboratory,
"age below 42 days?" in the next). When that threshold moves, every
prediction on the wrong side of it jumps. A linear regression changes its
weights a little; a tree changes its mind.

## 3. ➕ Going further: what averaging does

A single laboratory cannot ask 99 others for their models. But it can
imitate them: draw many random variations of **its own** 300 samples,
train a deep tree on each, and **average** their predictions. This is
called *bagging*, and module 6 is about it. Here we only look at what it
does to bias and spread.

🔮 **Predict first.** Compared with a single deep tree, the average of 25
deep trees will mainly have a lower… `"bias"` or `"variance"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["bias", "variance"])

In [ ]:
from sklearn.ensemble import BaggingRegressor

predictions["average of 25 deep trees"] = train_everywhere(
    BaggingRegressor(DecisionTreeRegressor(random_state=0), n_estimators=25, random_state=0),
    laboratories,
)
plot_disagreement(["deep tree", "average of 25 deep trees"])

In [ ]:
summary = pd.DataFrame({name: summarise(name) for name in predictions}).T
summary.round(2)

✏️ What did averaging reduce the most? Write `"bias"` or `"variance"`.

In [ ]:
averaging_reduces = ____
check("averaging_reduces", averaging_reduces)

The boxes shrink and stay where they were: the laboratories now agree
much more, around the same average as before. Averaging many models that
are each a little wrong in a different direction cancels part of their
disagreement. It cannot repair models that are all wrong in the **same**
direction.

scikit-learn's example
[Single estimator versus bagging](https://scikit-learn.org/1.8/auto_examples/ensemble/plot_bias_variance.html)
does this experiment on an artificial curve.

🧠 **Think.** Would averaging 25 **linear regressions** help as much? And
25 **shallow trees**? Use the table of section 2 to argue.

## 4. ➕ Going further: what more data does

So far each laboratory had 300 samples. What if each had 100, or 600?

🔮 **Predict first.** When the laboratories get more samples, which
number will go down clearly: the **bias of the linear regression** or
the **spread of the deep tree**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["bias of the linear regression", "spread of the deep tree"])

In [ ]:
rows = {}
for n_samples in [100, 300, 600]:
    labs = draw_laboratories(n_samples=n_samples)
    linear = train_everywhere(LinearRegression(), labs)
    deep = train_everywhere(DecisionTreeRegressor(random_state=0), labs)
    rows[n_samples] = {
        "bias of the linear regression": np.abs(linear.mean(axis=0) - truth).mean(),
        "spread of the deep tree": deep.std(axis=0).mean(),
    }
by_size = pd.DataFrame(rows).T.rename_axis("samples per laboratory")
by_size.round(2)

✏️ Which problem does collecting more data cure? Write `"bias"` or
`"variance"`.

In [ ]:
more_data_reduces = ____
check("more_data_reduces", more_data_reduces)

This is the learning curve of workbook M2.02 seen from the inside: more
samples kept helping the deep tree (its problem is variance) and did
nothing for the linear regression (its problem is bias).

One caution about our game: all the laboratories draw from the same pool
of 830 samples. With 600 samples each they have most of their samples in
common, and agree partly for that reason. The direction of the effect is
right; its size is exaggerated.

## 5. ➕ Going further: the error nobody can remove

Suppose we had the perfect model and endless data. Would the error be
zero? Look for samples that have **exactly the same recipe and the same
age**:

In [ ]:
features = list(data.columns)
twins = concrete.groupby(features)["strength"].agg(["count", "min", "max"])
twins = twins[twins["max"] > twins["min"]]
twins["difference"] = twins["max"] - twins["min"]
print(f"{len(twins)} groups of identical samples with different strengths")
twins.sort_values("difference", ascending=False).head(5).reset_index()[
    ["cement", "water", "age", "count", "min", "max", "difference"]
]

Same ingredients, same age, different strength. Concrete is not perfectly
reproducible: the mixing, the temperature of the day, the press all play
a part, and none of them is in our eight columns. This is **noise**.

✏️ Two samples have identical features. One broke at 30 MPa, the other
at 40 MPa. A model sees the same features twice, so it must give the
**same prediction** for both. What is the **smallest** mean absolute
error it can reach on these two samples? Try a prediction of 30, of 35,
of 40.

In [ ]:
smallest_possible_error = ____
check("smallest_possible_error", smallest_possible_error, hint="For each candidate prediction, average the two absolute errors.")

Every prediction error is made of three parts:

| part | comes from | what reduces it |
|---|---|---|
| **bias** | a model too rigid for the pattern | a more flexible model, better features |
| **variance** | a model that depends too much on its training set | more data, a simpler model, averaging |
| **noise** | what the features do not tell | new features (measure what is missing) |

🧠 **Think.**

1. In workbook M2.01 the fully grown tree had a training error of
   0.11 MPa, not 0. Which of the three parts explains it?
2. The company wants the error below 1 MPa. From what you have seen in
   the three workbooks of this module, is a better algorithm likely to be
   enough? What would you ask the laboratory to record?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the box plots of section 2: where you see bias, where you see
     variance, and why a small tree is not a stable model;
   - what averaging 25 trees changed and what it left untouched;
   - why more data helps one kind of model and not the other;
   - the twin samples: why no model can reach an error of zero.